# OpenAlex Citation Network Ingestion & DataLoader

This notebook demonstrates:
1. **API Ingestion** from OpenAlex (200M+ works, completely free, no API key required)
2. **Citation Network Analysis** and visualization
3. **Factory-based DataLoader Generator** (proper Design Pattern with distinct paths)
4. **Document Preparation** with rich metadata (arXiv/APA/MLA format)
5. **Section Labeling Interface** for optional ML-based section classification

---

## Why OpenAlex?

**OpenAlex** is a free, open-access, comprehensive index of scholarly literature maintained by the Allen Institute for AI (creators of Semantic Scholar). It provides:
- **200M+ works** (papers, preprints, datasets, books)
- **500M+ citation edges** (comprehensive citation graph)
- **100M+ authors** with affiliations and collaborative networks
- **No authentication required** for public queries
- **Unlimited free requests** (HTTP 429 rate-limiting only at extreme volumes)
- **Superior coverage** including preprints, reports, datasets (not just peer-reviewed)
- **Better discoverability** than Semantic Scholar for non-traditional research

See **Appendix: Academic Literature APIs** at the bottom for comparison with other services.

## Part 0: Dependencies & Setup

In [1]:
# Install required packages
import subprocess
import sys

packages = [
    'requests',
    'pandas',
    'numpy',
    'networkx',
    'matplotlib',
    'seaborn',
    'python-dateutil',
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ All dependencies installed")

✓ All dependencies installed


In [2]:
import requests
import json
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Iterator, Callable, Optional, Any
from dataclasses import dataclass, asdict
from pathlib import Path
from datetime import datetime
import time
from functools import wraps
import hashlib
import pickle

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All imports successful")

✓ All imports successful


## Part 1: OpenAlex API Client

In [3]:
@dataclass
class OpenAlexConfig:
    """Configuration for OpenAlex API access (completely free, no API key required)."""
    base_url: str = "https://api.openalex.org"
    works_endpoint: str = "/works"
    request_timeout: int = 30
    max_retries: int = 3
    retry_delay: float = 1.0
    email: str = ""  # Optional: provide email for better rate limiting courtesy

class OpenAlexClient:
    """Client for OpenAlex API (completely free, no authentication required).
    
    OpenAlex provides access to 200M+ scholarly works with 500M+ citation edges.
    No API key, no rate limiting for normal use.
    """
    
    def __init__(self, config: OpenAlexConfig = None):
        self.config = config or OpenAlexConfig()
    
    def search_works(
        self, 
        query: str, 
        limit: int = 50,
        offset: int = 0
    ) -> Dict[str, Any]:
        """
        Search for works in OpenAlex.
        
        Args:
            query: Search query (title/abstract keywords)
            limit: Max results per request (1-200)
            offset: Pagination offset
        
        Returns:
            Dict with 'results' (list of works), 'meta' (pagination info)
        """
        page = (offset // min(limit, 200)) + 1
        
        params = {
            'search': query,
            'per-page': min(limit, 200),
            'page': page,
        }
        
        if self.config.email:
            params['mailto'] = self.config.email
        
        url = f"{self.config.base_url}{self.config.works_endpoint}"
        
        for attempt in range(self.config.max_retries):
            try:
                response = requests.get(
                    url, 
                    params=params,
                    timeout=self.config.request_timeout
                )
                response.raise_for_status()
                return response.json()
            except requests.exceptions.RequestException as e:
                if attempt < self.config.max_retries - 1:
                    time.sleep(self.config.retry_delay * (2 ** attempt))
                else:
                    raise
    
    def get_work(
        self,
        work_id: str
    ) -> Dict[str, Any]:
        """
        Fetch detailed work information by OpenAlex ID.
        
        Args:
            work_id: OpenAlex work ID (e.g., "W2741809807") or DOI
        
        Returns:
            Work object with all available fields
        """
        params = {}
        if self.config.email:
            params['mailto'] = self.config.email
        
        url = f"{self.config.base_url}{self.config.works_endpoint}/{work_id}"
        
        for attempt in range(self.config.max_retries):
            try:
                response = requests.get(
                    url,
                    params=params,
                    timeout=self.config.request_timeout
                )
                response.raise_for_status()
                return response.json()
            except requests.exceptions.RequestException as e:
                if attempt < self.config.max_retries - 1:
                    time.sleep(self.config.retry_delay * (2 ** attempt))
                else:
                    raise

# Initialize client
openalex_client = OpenAlexClient()
print("✓ OpenAlex client initialized (free, no API key required)")

✓ OpenAlex client initialized (free, no API key required)


## Part 2: Download & Caching Functions

In [4]:
def create_cache_dir(base_dir: str = "./openalex_cache") -> Path:
    """Create cache directory for downloaded works."""
    cache_path = Path(base_dir)
    cache_path.mkdir(parents=True, exist_ok=True)
    return cache_path

CACHE_DIR = create_cache_dir()

def _hash_query(query: str) -> str:
    """Generate hash of query for cache key."""
    return hashlib.md5(query.encode()).hexdigest()[:12]

def download_works_from_openalex(
    query: str,
    limit: int = 50,
    client: OpenAlexClient = None
) -> List[Dict[str, Any]]:
    """
    Download works from OpenAlex API.
    
    Args:
        query: Search query
        limit: Max works to fetch
        client: OpenAlexClient instance
    
    Returns:
        List of work dictionaries
    """
    client = client or openalex_client
    print(f"📥 Downloading works for query: '{query}'")
    
    all_works = []
    offset = 0
    remaining = limit
    
    while remaining > 0:
        batch_size = min(remaining, 200)  # OpenAlex max per request
        result = client.search_works(query, limit=batch_size, offset=offset)
        
        works = result.get('results', [])
        if not works:
            break
        
        all_works.extend(works)
        remaining -= len(works)
        offset += len(works)
        
        print(f"  ✓ Fetched {len(all_works)} works so far")
        time.sleep(0.3)  # Respectful rate limiting
    
    print(f"✓ Downloaded {len(all_works)} works")
    return all_works

def cache_works(
    works: List[Dict[str, Any]],
    query: str,
    cache_dir: Path = CACHE_DIR
) -> Path:
    """
    Cache works to disk as pickle.
    
    Args:
        works: List of work dictionaries
        query: Original search query (for cache naming)
        cache_dir: Directory to save cache
    
    Returns:
        Path to cached file
    """
    cache_key = _hash_query(query)
    cache_file = cache_dir / f"works_{cache_key}.pkl"
    
    with open(cache_file, 'wb') as f:
        pickle.dump({
            'query': query,
            'works': works,
            'cached_at': datetime.now().isoformat(),
            'count': len(works)
        }, f)
    
    print(f"💾 Cached {len(works)} works to {cache_file}")
    return cache_file

def load_works_from_cache(
    query: str,
    cache_dir: Path = CACHE_DIR
) -> Optional[List[Dict[str, Any]]]:
    """
    Load works from cache if available.
    
    Args:
        query: Search query
        cache_dir: Directory containing cache
    
    Returns:
        List of works or None if not cached
    """
    cache_key = _hash_query(query)
    cache_file = cache_dir / f"works_{cache_key}.pkl"
    
    if cache_file.exists():
        with open(cache_file, 'rb') as f:
            data = pickle.load(f)
        print(f"📂 Loaded {data['count']} works from cache ({cache_file.name})")
        return data['works']
    
    return None

print("✓ Cache functions defined")

✓ Cache functions defined


## Part 3: Document Preparation & Section Labeling Interface

In [5]:
# ============================================================================
# SECTION LABELING INTERFACE (Sketch for ML-based classification)
# ============================================================================
# def label_section(section: Dict, model_fn: Optional[Callable] = None) -> Dict:
#   """
#   Optional: Pass section through model to assign generic/canonical labels.
#   
#   Args:
#     section: {"title": str, "text": str}
#     model_fn: Callable that classifies section type
#               e.g., model_fn(section) -> {"title": ..., "text": ..., 
#                                           "labels": ["background", "motivation"]}
#                                    or {"canonical_section": "introduction", ...}
#   
#   Returns:
#     section dict with optional "labels" or "canonical_section" key
#   
#   Example model_fn logic:
#     if "introduction" in section["title"].lower():
#       return {**section, "labels": ["background", "motivation", "related_work"]}
#     elif "method" in section["title"].lower():
#       return {**section, "canonical_section": "methods"}
#   """
#   if model_fn:
#     return model_fn(section)
#   return section

def prepare_document_with_sections(
    work: Dict[str, Any],
    citations: List[Dict[str, Any]] = None,
    section_labeler_fn: Optional[Callable] = None
) -> Dict[str, Any]:
    """
    Prepare document with sections (simulated from abstract/title/references).
    In production, would parse full-text PDF/.md to extract actual sections.
    
    Args:
        work: Work dict from OpenAlex API
        citations: List of cited work dicts
        section_labeler_fn: Optional function to label sections
                           e.g., label_section(section, model_fn)
    
    Returns:
        Document with sections structure
    """
    citations = citations or []
    
    # Extract publication info
    primary_location = work.get('primary_location', {})
    biblio = primary_location.get('is_accepted', False)
    
    # Construct metadata in arXiv/APA format
    metadata = {
        'work_id': work.get('id', ''),
        'title': work.get('title', 'Untitled'),
        'authors': [
            author.get('author', {}).get('display_name', 'Unknown')
            for author in work.get('authorships', [])
        ],
        'year': work.get('publication_year', None),
        'publication_date': work.get('publication_date', None),
        'abstract': work.get('abstract_inverted_index', {}),  # OpenAlex returns inverted index
        'abstract_text': reconstruct_abstract(work.get('abstract_inverted_index', {})),
        'venue': primary_location.get('source', {}).get('display_name', ''),
        'journal': primary_location.get('source', {}).get('display_name', ''),
        'doi': work.get('doi', ''),
        'url': work.get('url', ''),
        'citation_count': work.get('cited_by_count', 0),
        'is_open_access': work.get('is_open_access', False),
        'concepts': [c.get('display_name', '') for c in work.get('concepts', [])[:5]],
    }
    
    # Construct citation list (extracting key info)
    citation_list = [
        {
            'work_id': cit.get('id', ''),
            'title': cit.get('title', ''),
            'year': cit.get('publication_year', None),
            'authors': [
                author.get('author', {}).get('display_name', '')
                for author in cit.get('authorships', [])
            ],
        }
        for cit in citations
    ]
    
    # Construct pseudo-sections from abstract and metadata
    # (In production, parse actual PDF/markdown sections)
    abstract = metadata['abstract_text'] or ''
    
    sections = [
        {
            'title': 'Abstract',
            'text': abstract
        },
        {
            'title': 'Introduction',
            'text': f"Paper on topic: {metadata['title']}. Published in {metadata['venue']} ({metadata['year']}). Concepts: {', '.join(metadata['concepts'][:3])}"
        },
        {
            'title': 'References',
            'text': f"This work cites {len(citation_list)} references. Key citations include: " + 
                   "; ".join([f"{c['title']} ({c['year']})" for c in citation_list[:5]])
        }
    ]
    
    # Optional: Apply section labeling function
    # if section_labeler_fn:
    #     sections = [section_labeler_fn(sec) for sec in sections]
    
    return {
        'paper': {
            'sections': sections
        },
        'citations': citation_list,
        'metadata': metadata
    }

def prepare_document_flat(
    work: Dict[str, Any],
    citations: List[Dict[str, Any]] = None,
    section_labeler_fn: Optional[Callable] = None
) -> Dict[str, Any]:
    """
    Prepare document with flat text (no sections).
    
    Args:
        work: Work dict from OpenAlex API
        citations: List of cited work dicts
        section_labeler_fn: (unused in flat format)
    
    Returns:
        Document with concatenated text
    """
    citations = citations or []
    
    # Same metadata as sections version
    primary_location = work.get('primary_location', {})
    
    metadata = {
        'work_id': work.get('id', ''),
        'title': work.get('title', 'Untitled'),
        'authors': [
            author.get('author', {}).get('display_name', 'Unknown')
            for author in work.get('authorships', [])
        ],
        'year': work.get('publication_year', None),
        'publication_date': work.get('publication_date', None),
        'abstract_text': reconstruct_abstract(work.get('abstract_inverted_index', {})),
        'venue': primary_location.get('source', {}).get('display_name', ''),
        'journal': primary_location.get('source', {}).get('display_name', ''),
        'doi': work.get('doi', ''),
        'url': work.get('url', ''),
        'citation_count': work.get('cited_by_count', 0),
        'is_open_access': work.get('is_open_access', False),
        'concepts': [c.get('display_name', '') for c in work.get('concepts', [])[:5]],
    }
    
    citation_list = [
        {
            'work_id': cit.get('id', ''),
            'title': cit.get('title', ''),
            'year': cit.get('publication_year', None),
            'authors': [
                author.get('author', {}).get('display_name', '')
                for author in cit.get('authorships', [])
            ],
        }
        for cit in citations
    ]
    
    # Concatenate full text
    full_text = f"""
{metadata['title']}

Authors: {', '.join(metadata['authors'][:5])}
Year: {metadata['year']}
Venue: {metadata['venue']}
Open Access: {metadata['is_open_access']}

Abstract:
{metadata['abstract_text']}

References ({len(citation_list)} total):
{chr(10).join([f"- {c['title']} ({c['year']})" for c in citation_list[:10]])}
    """
    
    return {
        'paper': full_text,  # Flat text, not dict
        'citations': citation_list,
        'metadata': metadata
    }

def reconstruct_abstract(abstract_inverted_index: Dict) -> str:
    """
    Reconstruct abstract from OpenAlex inverted index format.
    OpenAlex returns abstract as {"word": [positions]}, need to reconstruct.
    
    Args:
        abstract_inverted_index: Inverted index dict from OpenAlex
    
    Returns:
        Reconstructed abstract string
    """
    if not abstract_inverted_index:
        return ""
    
    # Build position -> word mapping
    position_map = {}
    for word, positions in abstract_inverted_index.items():
        for pos in positions:
            position_map[pos] = word
    
    # Reconstruct in order
    abstract_words = [position_map[i] for i in sorted(position_map.keys())]
    return ' '.join(abstract_words)

print("✓ Document preparation functions defined")

✓ Document preparation functions defined


## Part 4: Factory Design Pattern - Distinct Generator Paths

In [6]:
# ============================================================================
# PATH 1: Generator WITH CACHING
# ============================================================================
def _generator_with_caching(
    works: List[Dict[str, Any]],
    cache_fn: Callable,
    prepare_doc_fn: Callable,
    section_labeler_fn: Optional[Callable],
    fetch_references: bool,
    client: 'OpenAlexClient'
) -> Iterator[Dict[str, Any]]:
    """
    PRIMARY GENERATOR (PATH 1): Yields single docs WITH caching side-effect.
    
    Distinct path: Each document is cached immediately after preparation.
    
    Yields:
        Single prepared document dict (with side-effect of caching)
    """
    for work in works:
        # Prepare document
        citations = []
        if fetch_references and work.get('referenced_works'):
            citations = work['referenced_works'][:10]
        
        doc = prepare_doc_fn(
            work=work,
            citations=citations,
            section_labeler_fn=section_labeler_fn
        )
        
        # CACHING SIDE-EFFECT (Path 1 only)
        cache_fn([doc])
        
        yield doc

# ============================================================================
# PATH 2: Generator WITHOUT CACHING
# ============================================================================
def _generator_without_caching(
    works: List[Dict[str, Any]],
    prepare_doc_fn: Callable,
    section_labeler_fn: Optional[Callable],
    fetch_references: bool,
    client: 'OpenAlexClient'
) -> Iterator[Dict[str, Any]]:
    """
    PRIMARY GENERATOR (PATH 2): Yields single docs WITHOUT caching.
    
    Distinct path: Pure preparation, no caching logic.
    
    Yields:
        Single prepared document dict (no side-effects)
    """
    for work in works:
        # Prepare document
        citations = []
        if fetch_references and work.get('referenced_works'):
            citations = work['referenced_works'][:10]
        
        doc = prepare_doc_fn(
            work=work,
            citations=citations,
            section_labeler_fn=section_labeler_fn
        )
        
        # NO CACHING (Path 2 only)
        yield doc

# ============================================================================
# FACTORY: Routes to appropriate path and returns initialized generator
# ============================================================================
def work_generator_factory(
    works: List[Dict[str, Any]],
    prepare_doc_fn: Callable = prepare_document_with_sections,
    section_labeler_fn: Optional[Callable] = None,
    fetch_references: bool = False,
    client: Optional['OpenAlexClient'] = None,
    cache_fn: Optional[Callable] = None,
) -> Iterator[Dict[str, Any]]:
    """
    FACTORY (Design Pattern): Produces and returns initialized generator.
    
    Uses TWO DISTINCT INITIALIZATION PATHS based on presence of cache_fn:
    - PATH 1 (with cache): _generator_with_caching()
    - PATH 2 (without cache): _generator_without_caching()
    
    Each path has physically separate code, control flow, and side-effects.
    
    Args:
        works: List of raw work dicts from OpenAlex API
        prepare_doc_fn: Function to prepare/format document
        section_labeler_fn: Optional function to label sections
        fetch_references: Whether to fetch reference data
        client: OpenAlexClient instance
        cache_fn: Caching function.
                 - If provided: returns generator with caching (PATH 1)
                 - If None: returns generator without caching (PATH 2)
    
    Returns:
        Initialized generator from PATH 1 or PATH 2
    """
    client = client or openalex_client
    
    # FACTORY DECISION POINT: Route to appropriate path
    if cache_fn is not None:
        # PATH 1: WITH CACHING
        print("🏭 Factory: Initializing generator WITH caching (Path 1)")
        return _generator_with_caching(
            works=works,
            cache_fn=cache_fn,
            prepare_doc_fn=prepare_doc_fn,
            section_labeler_fn=section_labeler_fn,
            fetch_references=fetch_references,
            client=client
        )
    else:
        # PATH 2: WITHOUT CACHING
        print("🏭 Factory: Initializing generator WITHOUT caching (Path 2)")
        return _generator_without_caching(
            works=works,
            prepare_doc_fn=prepare_doc_fn,
            section_labeler_fn=section_labeler_fn,
            fetch_references=fetch_references,
            client=client
        )

print("✓ Factory pattern implemented (two distinct paths)")

✓ Factory pattern implemented (two distinct paths)


## Part 5: Batch Composer Function

In [7]:
def batch_generator(
    primary_generator: Iterator[Dict[str, Any]],
    batch_size: int = 32
) -> Iterator[List[Dict[str, Any]]]:
    """
    Comprising function that yields BATCHES from the primary generator.
    
    Composes the primary generator (from factory) into batch groups.
    Works with either path (with or without caching).
    
    Args:
        primary_generator: Generator from work_generator_factory (either path)
        batch_size: Number of documents per batch
    
    Yields:
        List of documents (batch)
    """
    batch = []
    for doc in primary_generator:
        batch.append(doc)
        if len(batch) == batch_size:
            yield batch
            batch = []
    
    # Yield remaining documents if batch not full
    if batch:
        yield batch

print("✓ Batch composer function defined")

✓ Batch composer function defined


## Part 6: Demo - Download Works

In [8]:
# Download works from OpenAlex
DEMO_QUERY = "graph neural networks"
DEMO_LIMIT = 10  # Small limit for demo

# Try loading from cache first
works = load_works_from_cache(DEMO_QUERY)

if works is None:
    # Download and cache
    works = download_works_from_openalex(DEMO_QUERY, limit=DEMO_LIMIT, client=openalex_client)
    cache_works(works, DEMO_QUERY)

print(f"\n📊 Downloaded {len(works)} works")
if works:
    print(f"\nFirst work example:")
    print(json.dumps({
        'id': works[0].get('id'),
        'title': works[0].get('title'),
        'year': works[0].get('publication_year'),
        'cited_by_count': works[0].get('cited_by_count'),
        'is_open_access': works[0].get('is_open_access')
    }, indent=2))

📥 Downloading works for query: 'graph neural networks'
  ✓ Fetched 10 works so far
✓ Downloaded 10 works
💾 Cached 10 works to openalex_cache/works_ee5de8831670.pkl

📊 Downloaded 10 works

First work example:
{
  "id": "https://openalex.org/W2116341502",
  "title": "The Graph Neural Network Model",
  "year": 2008,
  "cited_by_count": 9183,
  "is_open_access": null
}


## Part 7: Test PATH 1 - Generator WITH Caching

In [9]:
if works:
    print("\n" + "="*70)
    print("TEST: Factory → PATH 1 (WITH CACHING)")
    print("="*70)
    
    # Factory returns PATH 1 generator (with caching)
    primary_gen_cached = work_generator_factory(
        works=works[:3],
        prepare_doc_fn=prepare_document_with_sections,
        fetch_references=True,
        client=openalex_client,
        cache_fn=cache_works  # <-- Triggers PATH 1 (with caching)
    )
    
    # Consume generator
    for idx, doc in enumerate(primary_gen_cached, 1):
        print(f"\n[{idx}] {doc['metadata']['title'][:60]}...")
        print(f"    Sections: {len(doc['paper']['sections'])}, Citations: {len(doc['citations'])}")
        print(f"    (Document was cached during generation)")
else:
    print("⚠️  No works available")


TEST: Factory → PATH 1 (WITH CACHING)
🏭 Factory: Initializing generator WITH caching (Path 1)


AttributeError: 'str' object has no attribute 'get'

## Part 8: Test PATH 2 - Generator WITHOUT Caching

In [ ]:
if works:
    print("\n" + "="*70)
    print("TEST: Factory → PATH 2 (WITHOUT CACHING)")
    print("="*70)
    
    # Factory returns PATH 2 generator (no caching)
    primary_gen_nocache = work_generator_factory(
        works=works[:3],
        prepare_doc_fn=prepare_document_with_sections,
        fetch_references=True,
        client=openalex_client,
        cache_fn=None  # <-- Triggers PATH 2 (no caching)
    )
    
    # Consume generator
    for idx, doc in enumerate(primary_gen_nocache, 1):
        print(f"\n[{idx}] {doc['metadata']['title'][:60]}...")
        print(f"    Sections: {len(doc['paper']['sections'])}, Citations: {len(doc['citations'])}")
        print(f"    (Document was NOT cached - pure generation)")
else:
    print("⚠️  No works available")

## Part 9: Test Batch Composer with Caching

In [ ]:
if works:
    print("\n" + "="*70)
    print("TEST: Factory (PATH 1) → Batch Composer")
    print("="*70)
    
    # Factory → PATH 1 with caching
    primary_gen = work_generator_factory(
        works=works,
        prepare_doc_fn=prepare_document_with_sections,
        fetch_references=True,
        client=openalex_client,
        cache_fn=cache_works
    )
    
    # Compose into batches
    batch_gen = batch_generator(primary_gen, batch_size=3)
    
    # Consume batches
    for batch_idx, batch in enumerate(batch_gen, 1):
        print(f"\n📦 Batch {batch_idx} ({len(batch)} documents):")
        for doc_idx, doc in enumerate(batch, 1):
            print(f"  [{doc_idx}] {doc['metadata']['title'][:50]}...")
else:
    print("⚠️  No works available")

## Part 10: Test Flat Format

In [ ]:
if works and len(works) >= 2:
    print("\n" + "="*70)
    print("TEST: Factory (PATH 2) with FLAT format")
    print("="*70)
    
    # Factory → PATH 2 (no caching) with flat format
    primary_gen_flat = work_generator_factory(
        works=works[:2],
        prepare_doc_fn=prepare_document_flat,  # <-- Flat format
        fetch_references=True,
        client=openalex_client,
        cache_fn=None  # <-- PATH 2 (no caching)
    )
    
    flat_doc = next(primary_gen_flat)
    
    print("\n📄 Single Document (flat text format):")
    print(f"\nMetadata:")
    for key in ['title', 'authors', 'year', 'venue', 'is_open_access']:
        print(f"  {key}: {flat_doc['metadata'].get(key)}")
    
    print(f"\nPaper Text ({len(flat_doc['paper'])} chars):")
    print(flat_doc['paper'][:400])
    print("...")
else:
    print("⚠️  Not enough works for test")

## Part 11: Citation Network Analysis

In [ ]:
if works:
    # Build citation network from downloaded works
    G = nx.DiGraph()
    
    # Add nodes (works)
    for work in works:
        G.add_node(
            work['id'],
            title=work.get('title', 'Unknown'),
            year=work.get('publication_year', None),
            citations=work.get('cited_by_count', 0)
        )
    
    # Add edges (citations between works in our set)
    work_ids = {w['id'] for w in works}
    
    for work in works:
        for ref_id in work.get('referenced_works', [])[:5]:  # Limit to first 5 refs
            if ref_id in work_ids:
                G.add_edge(work['id'], ref_id, weight=1)
    
    print(f"📊 Citation Network:")
    print(f"  Nodes: {G.number_of_nodes()}")
    print(f"  Edges: {G.number_of_edges()}")
    if G.number_of_nodes() > 0:
        print(f"  Density: {nx.density(G):.4f}")
    
    # Statistics
    in_degrees = dict(G.in_degree())
    out_degrees = dict(G.out_degree())
    
    print(f"\n  Most cited works:")
    for work_id, in_deg in sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:3]:
        node_data = G.nodes[work_id]
        print(f"    - {node_data['title'][:50]}... (cited {in_deg} times)")
else:
    print("⚠️  No works available")

## Part 12: PyTorch Dataset Wrapper (Optional)

In [ ]:
try:
    import torch
    from torch.utils.data import Dataset, DataLoader
    
    class OpenAlexDocumentDataset(Dataset):
        """PyTorch Dataset wrapping factory-generated documents."""
        
        def __init__(
            self,
            works: List[Dict[str, Any]],
            prepare_doc_fn: Callable = prepare_document_with_sections,
            section_labeler_fn: Optional[Callable] = None,
            fetch_references: bool = False,
            use_caching: bool = False,
        ):
            # Use factory to generate documents (choose path based on use_caching)
            gen = work_generator_factory(
                works=works,
                prepare_doc_fn=prepare_doc_fn,
                section_labeler_fn=section_labeler_fn,
                fetch_references=fetch_references,
                cache_fn=cache_works if use_caching else None  # Factory routes to path
            )
            
            # Pre-generate all documents
            self.documents = list(gen)
        
        def __len__(self):
            return len(self.documents)
        
        def __getitem__(self, idx):
            return self.documents[idx]
    
    print("✓ PyTorch Dataset wrapper available")
    print("  Usage: dataset = OpenAlexDocumentDataset(works, use_caching=True/False)")
    print("         loader = DataLoader(dataset, batch_size=32)")

except ImportError:
    print("⚠️  PyTorch not installed (optional)")

## Part 13: Architecture Summary

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════╗
║           FACTORY DESIGN PATTERN ARCHITECTURE SUMMARY              ║
╚════════════════════════════════════════════════════════════════════╝

┌─ FACTORY: work_generator_factory()
│
├─→ PATH 1: _generator_with_caching()
│   │ Side-effect: Caches each document via cache_fn
│   │ Used when: cache_fn is provided (not None)
│   │ Example: work_generator_factory(..., cache_fn=cache_works)
│   └─→ Yields: Single prepared documents WITH caching
│
├─→ PATH 2: _generator_without_caching()
│   │ Side-effect: None (pure generation)
│   │ Used when: cache_fn is None
│   │ Example: work_generator_factory(..., cache_fn=None)
│   └─→ Yields: Single prepared documents WITHOUT caching
│
└─→ Both paths feed into: batch_generator()
    └─→ Composes primary generator into batches


TWO DISTINCT INITIALIZATION PATHS:

┌─────────────────────────────────────────────────────────────┐
│ Path 1: WITH CACHING                                        │
├─────────────────────────────────────────────────────────────┤
│ factory = work_generator_factory(                           │
│     works=works,                                            │
│     cache_fn=cache_works   # ← Caching enabled             │
│ )                                                           │
│                                                             │
│ for doc in factory:                                         │
│     print(doc)  # Doc is cached as side-effect            │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│ Path 2: WITHOUT CACHING                                     │
├─────────────────────────────────────────────────────────────┤
│ factory = work_generator_factory(                           │
│     works=works,                                            │
│     cache_fn=None   # ← Caching disabled                   │
│ )                                                           │
│                                                             │
│ for doc in factory:                                         │
│     print(doc)  # Doc is NOT cached (pure generation)      │
└─────────────────────────────────────────────────────────────┘


COMPOSING WITH BATCH GENERATOR:

# Use either path, compose into batches
primary = work_generator_factory(works, cache_fn=cache_works)  # Or None
batches = batch_generator(primary, batch_size=32)

for batch in batches:
    # batch is a list of 32 documents
    process(batch)


KEY DIFFERENCES:

Criterion          │ Path 1 (Caching)      │ Path 2 (No Cache)
──────────────────┼────────────────────────┼─────────────────────
Code Path          │ _generator_with_...   │ _generator_without_...
Caching Logic      │ Present + Active      │ Absent
Side-effects       │ Yes (caching)         │ None
Memory Usage       │ Disk (cache file)     │ Memory only
Function Called    │ cache_fn per doc      │ None
Use Case           │ Persistence needed    │ In-memory only
""")

---

# Appendix: Academic Literature APIs Comparison

## Overview Table

| Service | Coverage | Cost | Auth Required | Rate Limit | Best For |
|---------|----------|------|-------|---------|----------|
| **OpenAlex** | 200M works, 500M citations | **FREE** | **No** | Generous (HTTP 429 only at extreme volumes) | Large-scale research, no budget constraints |
| **Semantic Scholar** | 200M papers, 2.4B citations | **FREE** | No | Public API: standard rate limits | Peer-reviewed focus, detailed metadata |
| **Crossref** | 140M+ metadata records | **FREE** | No | Moderate (~50 req/sec) | DOI resolution, structured metadata |
| **arXiv** | 2.3M papers, full text | **FREE** | No | Moderate (~3 req/sec) | Preprints, CS/Physics/Math |
| **CORE** | 400M+ scholarly documents | **FREE** | No (bulk) | Variable | Open access content, preservation |
| **PubMed** | 35M+ biomedical articles | **FREE** | No | Moderate (~3 req/sec) | Biomedical/life sciences |
| **Web of Science** | 90M+ (selective) | **PAID** (~$15K-30K/yr) | Yes (key) | Strict | Citation impact, JCR metrics |
| **Scopus** | 95M+ (selective) | **PAID** (~$10K-40K/yr) | Yes (key) | Strict | Citation impact, broad coverage |
| **ProQuest Dissertations** | 5M+ theses/dissertations | **PAID** (institutional) | Yes | Limited | Theses, dissertations |

---

## Detailed Descriptions

### Free, Large-Scale APIs

#### **OpenAlex** ⭐ (Used in this notebook)
- **Coverage**: 200M+ scholarly works (papers, preprints, datasets, books, reports)
- **Citation Graph**: 500M+ citation edges
- **Cost**: Completely FREE, no API key required
- **Authentication**: None
- **Rate Limiting**: No practical limits for reasonable use; HTTP 429 only at extreme volumes
- **Best for**: Large-scale citation network analysis, comprehensive coverage beyond peer-reviewed
- **Strengths**:
  - Largest freely accessible index
  - Includes preprints, reports, datasets (not just journals)
  - Rich author/institution affiliation data
  - Funding information for some works
  - Maintained by Allen Institute for AI (credible)
- **Limitations**:
  - Abstract only available as inverted index (requires reconstruction)
  - No full-text access
  - Citation type classification limited

#### **Semantic Scholar**
- **Coverage**: 200M+ papers (peer-reviewed focus), 2.4B+ citation edges
- **Cost**: FREE public API (no key required)
- **Rate Limiting**: Standard (reasonable for research)
- **Best for**: Peer-reviewed literature focus, AI-enhanced features (TLDRs, embeddings)
- **Strengths**:
  - Full abstract text (not inverted index)
  - AI-generated paper summaries (TLDRs)
  - SPECTER embeddings for semantic search
  - Citation intent classification
  - Influential citation detection
- **Limitations**:
  - Smaller than OpenAlex for non-traditional research
  - Less coverage of preprints/datasets

#### **Crossref**
- **Coverage**: 140M+ metadata records (focus on published, registered DOIs)
- **Cost**: FREE
- **Best for**: DOI-based lookup, structured metadata, publisher information
- **Rate Limit**: ~50 requests/second (good)
- **Strengths**:
  - Official DOI metadata registry
  - Rich publication metadata (funding, licenses)
  - Funders database
  - Reference metadata (cited-by links)
- **Limitations**:
  - Metadata only (no abstracts typically)
  - Focus on published/registered DOIs

#### **arXiv**
- **Coverage**: 2.3M papers (Computer Science, Physics, Mathematics, Biology, Finance, Statistics)
- **Cost**: FREE
- **Best for**: Preprints in CS/Physics/Math, full-text PDFs available
- **Rate Limit**: ~3 requests/second (moderate)
- **Strengths**:
  - Full-text PDF/source available
  - No embargo periods
  - Large volume in CS/Physics
  - Clean metadata
- **Limitations**:
  - Limited to specific fields
  - Preprints only (limited peer-review data)
  - Citation metadata less comprehensive

#### **CORE**
- **Coverage**: 400M+ open access documents (aggregated from repositories)
- **Cost**: FREE (bulk downloads available)
- **Best for**: Open access content, research preservation, diverse sources
- **Strengths**:
  - Aggregates from 10K+ repositories
  - Includes non-traditional research (theses, reports)
  - Full-text indexing in many cases
- **Limitations**:
  - Quality varies (aggregated from many sources)
  - Citation graph less mature
  - Deduplication challenges

#### **PubMed**
- **Coverage**: 35M+ biomedical/life sciences articles
- **Cost**: FREE
- **Best for**: Biomedical literature, medical research
- **Rate Limit**: ~3 requests/second
- **Strengths**:
  - Comprehensive biomedical coverage
  - Mesh indexing (controlled vocabulary)
  - Links to full-text via PMC
- **Limitations**:
  - Biomedical-only
  - Citation graph limited

---

### Paid, Commercial APIs

#### **Web of Science** (Clarivate)
- **Cost**: $15,000-30,000+/year (institutional licenses)
- **Coverage**: 90M+ (highly selective, peer-reviewed focus)
- **Best for**: Citation impact metrics, JCR (Journal Citation Reports), institutional research
- **Strengths**:
  - Gold standard for citation impact (IF, h-index)
  - Rigorous curation (high quality)
  - Research areas, document types carefully classified
  - FWCI (field-weighted citation impact)
- **Limitations**:
  - Very expensive
  - Smaller coverage than open alternatives
  - Limited to institutions with subscriptions

#### **Scopus** (Elsevier)
- **Cost**: $10,000-40,000+/year (institutional licenses)
- **Coverage**: 95M+ (broader than WoS, but still selective)
- **Best for**: Citation metrics, author h-index, broader multidisciplinary coverage
- **Strengths**:
  - Broader than WoS (includes more sources)
  - CiteScore metrics
  - SJR (Scimago Journal Ranking)
  - Author profiles with metrics
- **Limitations**:
  - Expensive
  - Institutional access typically required
  - Less transparent curation than WoS

#### **ProQuest Dissertations & Theses**
- **Cost**: Institutional subscription or per-document purchase
- **Coverage**: 5M+ theses, dissertations
- **Best for**: Academic theses, dissertations as research data

---

## Recommendation Matrix

**Choose OpenAlex if you want:**
- Largest freely accessible index
- No authentication/API key hassle
- Comprehensive coverage (papers + preprints + datasets + reports)
- Author/institution affiliation network
- No budget constraints ✓ (This notebook's choice)

**Choose Semantic Scholar if you want:**
- Peer-reviewed papers focus
- AI-enhanced summaries (TLDRs)
- Better abstract quality
- Citation intent classification
- Embeddings for semantic search

**Choose Crossref if you want:**
- DOI-based lookup
- Funding/funder information
- Published metadata only

**Choose arXiv if you want:**
- Preprints with full-text PDFs
- CS/Physics/Math focus
- No embargo periods

**Choose Web of Science/Scopus only if:**
- You have institutional access
- You need official citation impact metrics (JCR, CiteScore)
- Budget allows ($15K-40K+/year)

---

## Implementation Notes

- **This notebook uses OpenAlex** because:
  1. Largest dataset (200M+ works)
  2. Completely free with no API key
  3. No rate limiting for reasonable use
  4. Includes non-traditional research
  5. Excellent for educational/research purposes

- **To switch to another API**, replace the client and adapt the document preparation functions to the API's response format.
